# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [6]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [7]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [8]:
# 🤖 AGENT FUNCTION

import re
import logging
from datetime import datetime

# --- Bonus: Logging setup ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")


# --- Bonus: Additional tool (word counter) ---
def count_words(text: str) -> dict:
    """Count total and unique words in a sentence."""
    words = re.findall(r"\w+", text.lower())
    return {"total_words": len(words), "unique_words": len(set(words))}


def agent(query: str):
    """Route a user query to the appropriate tool based on intent.

    Returns a structured JSON-like dict:
      {"type": "calculation" | "keywords" | "word_count" | "general" | "error",
       "result": ...}
    """
    query_lower = query.lower()

    try:
        # --- Branch 1: Math queries -> Calculator tool ---
        if any(k in query_lower for k in ["calculate", "calculator", "compute", "solve"]):
            # Strip the instruction prefix so the remainder is a valid expression
            expr = re.sub(
                r"^(please\s+)?(calculate|calculator|compute|solve)\s+",
                "", query, flags=re.IGNORECASE
            ).strip()
            response = {"type": "calculation", "result": calculator(expr)}

        # --- Branch 2: Keyword queries -> Keyword extractor tool ---
        elif "keyword" in query_lower:
            # Strip the instruction prefix, keep the actual text
            text = re.sub(
                r"^(please\s+)?(extract\s+)?keywords?\s+(from\s+)?",
                "", query, flags=re.IGNORECASE
            ).strip()
            response = {"type": "keywords", "result": extract_keywords(text)}

        # --- Branch 3 (bonus): Word counting query -> extra tool ---
        elif any(k in query_lower for k in ["count words", "word count", "how many words"]):
            response = {"type": "word_count", "result": count_words(query)}

        # --- Branch 4: Everything else -> General response ---
        else:
            response = {
                "type": "general",
                "result": (
                    f"General assistant response: I received \"{query}\". "
                    "Try asking me to calculate something (e.g. \"Calculate 20 + 5\"), "
                    "extract keywords, or count words."
                ),
            }

    except Exception as e:
        # --- Basic error handling ---
        logger.error("Error while processing query %r: %s", query, e, exc_info=True)
        response = {"type": "error", "result": f"An unexpected error occurred: {e}"}

    # --- Bonus: Log every request ---
    logger.info("Query: %r -> type=%s", query, response["type"])
    return response

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [9]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-08-10 18:31:22,539 | INFO | Query: 'Calculate 20 + 5' -> type=calculation
2026-08-10 18:31:22,540 | INFO | Query: 'Extract keywords from Artificial Intelligence is transforming industries' -> type=keywords
2026-08-10 18:31:22,541 | INFO | Query: 'What is machine learning?' -> type=general


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['transforming', 'intelligence', 'industries', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'General assistant response: I received "What is machine learning?". Try asking me to calculate something (e.g. "Calculate 20 + 5"), extract keywords, or count words.'}
--------------------------------------------------


In [10]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

2026-08-10 18:32:33,413 | INFO | Query: 'calculate 1+2' -> type=calculation


Response: {'type': 'calculation', 'result': '3'}
